In [1]:
import torch
import torch.nn as nn
from torch.ao.quantization import get_default_qconfig_mapping
from torch.ao.quantization.quantize_fx import prepare_fx, convert_fx

# 1. Define standard full-precision baseline FP32 module
class InferenceBlock(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = nn.Conv2d(3, 16, kernel_size=3, padding=1)
        self.relu = nn.ReLU()
        self.fc = nn.Linear(16 * 32 * 32, 10)
        
    def forward(self, x):
        x = self.conv(x)
        x = self.relu(x)
        x = torch.flatten(x, 1)
        x = self.fc(x)
        return x

model_fp32 = InferenceBlock().eval()

# 2. Configure the Backend Mapping Target Architecture
# This sets per-channel symmetric weights and per-tensor asymmetric activations
qconfig_mapping = get_default_qconfig_mapping("x86") # Or 'qnnpack' for ARM edge setups

# 3. Instrument the Model Graph via Graph Module Transformation
# prepare_fx inserts calibration observers into the computational graph path
example_inputs = torch.randn(1, 3, 32, 32)
prepared_model = prepare_fx(model_fp32, qconfig_mapping, example_inputs)

print("--- Quantization Observers Inserted ---")
print(f"Prepared model trace tracking structures operational.\n")

# 4. INT8 Calibration Stage
# Run representative data through the model to map dynamic activation ranges
calibration_dataset = [torch.randn(1, 3, 32, 32) for _ in range(50)]
with torch.no_grad():
    for calibration_batch in calibration_dataset:
        # Observers actively record tensor range metrics during these forward passes
        prepared_model(calibration_batch)

print("--- Calibration Phase Concluded Success ---")
print("Tensors statistics successfully recorded across historical channels.\n")

# 5. Convert the Graph to Fixed-Point INT8 Quantized Layout
# convert_fx removes observers, calculates scales/zero-points, and replaces float layers
quantized_model = convert_fx(prepared_model)

print("--- Graph Structural Fusion Complete ---")
print(quantized_model)

# 6. Verify System Evaluation Output Run
with torch.no_grad():
    quantized_output = quantized_model(example_inputs)
    print(f"\nQuantized Inference Output Vector Shape: {quantized_output.shape}")

/tmp/ipykernel_1083/1656157388.py:30: DeprecationWarning: torch.ao.quantization is deprecated and will be removed in 2.10. 
For migrations of users: 
1. Eager mode quantization (torch.ao.quantization.quantize, torch.ao.quantization.quantize_dynamic), please migrate to use torchao eager mode quantize_ API instead 
2. FX graph mode quantization (torch.ao.quantization.quantize_fx.prepare_fx,torch.ao.quantization.quantize_fx.convert_fx, please migrate to use torchao pt2e quantization API instead (prepare_pt2e, convert_pt2e) 
3. pt2e quantization has been migrated to torchao (https://github.com/pytorch/ao/tree/main/torchao/quantization/pt2e) 
see https://github.com/pytorch/ao/issues/2259 for more details
  prepared_model = prepare_fx(model_fp32, qconfig_mapping, example_inputs)


--- Quantization Observers Inserted ---
Prepared model trace tracking structures operational.

--- Calibration Phase Concluded Success ---
Tensors statistics successfully recorded across historical channels.

--- Graph Structural Fusion Complete ---
GraphModule(
  (conv): QuantizedConvReLU2d(3, 16, kernel_size=(3, 3), stride=(1, 1), scale=0.01931193470954895, zero_point=0, padding=(1, 1))
  (fc): QuantizedLinear(in_features=16384, out_features=10, scale=0.010734331794083118, zero_point=70, qscheme=torch.per_channel_affine)
)



def forward(self, x):
    conv_input_scale_0 = self.conv_input_scale_0
    conv_input_zero_point_0 = self.conv_input_zero_point_0
    quantize_per_tensor = torch.quantize_per_tensor(x, conv_input_scale_0, conv_input_zero_point_0, torch.quint8);  x = conv_input_scale_0 = conv_input_zero_point_0 = None
    conv = self.conv(quantize_per_tensor);  quantize_per_tensor = None
    flatten = torch.flatten(conv, 1);  conv = None
    fc = self.fc(flatten);  flatten = None

/usr/local/lib/python3.11/site-packages/torch/ao/quantization/observer.py:1039: UserWarning: Please use quant_min and quant_max to specify the range for observers.                     reduce_range will be deprecated in a future release of PyTorch.
  super().__init__(
/tmp/ipykernel_1083/1656157388.py:48: DeprecationWarning: torch.ao.quantization is deprecated and will be removed in 2.10. 
For migrations of users: 
1. Eager mode quantization (torch.ao.quantization.quantize, torch.ao.quantization.quantize_dynamic), please migrate to use torchao eager mode quantize_ API instead 
2. FX graph mode quantization (torch.ao.quantization.quantize_fx.prepare_fx,torch.ao.quantization.quantize_fx.convert_fx, please migrate to use torchao pt2e quantization API instead (prepare_pt2e, convert_pt2e) 
3. pt2e quantization has been migrated to torchao (https://github.com/pytorch/ao/tree/main/torchao/quantization/pt2e) 
see https://github.com/pytorch/ao/issues/2259 for more details
  quantized_model = con